# RAG Assignment: Baseline (Bi-Encoder) vs. Re-Ranking (Cross-Encoder)

**Document:** NVIDIA Corporation *Annual Report / Form 10-K / Proxy Statement, fiscal year 2024* (187 pages: business description, risk factors, MD&A, financial statements, executive compensation, governance).

| Component | Choice |
|---|---|
| Bi-Encoder (dense retriever) | `BAAI/bge-small-en-v1.5`, cosine similarity (normalised vectors + FAISS inner product) |
| Cross-Encoder (re-ranker) | `cross-encoder/ms-marco-MiniLM-L-6-v2` |
| LLM | `qwen2.5:7b-instruct` run locally through Ollama (temperature 0, fixed seed) |

Both pipelines use the **same chunks, same index, same prompt template**; the only difference is *which 3 chunks* reach the LLM.

## 0. Setup

In [1]:
# pip install sentence-transformers pypdf faiss-cpu ollama pandas
# Ollama must be running with the model pulled:  ollama pull qwen2.5:7b-instruct
import os, re, time, json, random, urllib.request
import numpy as np, pandas as pd, faiss, pypdf, ollama
from sentence_transformers import SentenceTransformer, CrossEncoder

PDF_PATH = "data/nvidia_annual_report_fy2024.pdf"
PDF_URL  = "https://s201.q4cdn.com/141608511/files/doc_financials/2024/ar/NVIDIA-2024-Annual-Report.pdf"
LLM      = "qwen2.5:7b-instruct"
os.makedirs("data", exist_ok=True)
if not os.path.exists(PDF_PATH):           # ~35 MB download if the PDF is not already in the repo
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)

/home/mohith/miniconda3/envs/chestxray/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load and chunk the document
Text is cleaned (hyphenation across line breaks removed) and cut into **overlapping 180-word windows (40-word overlap)**. Each chunk remembers its start page so retrieved evidence can be traced back to the PDF. Pages that are mostly graphics or tables give little text, so 187 pages become roughly 98k words.

In [2]:
reader = pypdf.PdfReader(PDF_PATH)
words = []                                   # (word, page_number)
for i, page in enumerate(reader.pages):
    t = page.extract_text() or ""
    t = re.sub(r"-\n(?=[a-z])", "", t)       # re-join words hyphenated at line end
    t = re.sub(r"\s+", " ", t)
    words += [(w, i + 1) for w in t.split()]

CHUNK, OVERLAP = 180, 40
chunks = []
for start in range(0, len(words), CHUNK - OVERLAP):
    seg = words[start:start + CHUNK]
    if len(seg) < 40:                         # drop a tiny trailing fragment
        break
    chunks.append({"id": len(chunks), "page": seg[0][1], "text": " ".join(w for w, _ in seg)})

print(f"{len(reader.pages)} pages -> {len(words)} words -> {len(chunks)} chunks")
print(chunks[456]["text"][:300], "...")

187 pages -> 97933 words -> 700 chunks
from us, and our revenue could be adversely affected if we lose or are prevented from selling to any of these customers. We receive a significant amount of our revenue from a limited number of customers within our distribution and partner network. Sales to one customer, Customer A, represented 13% o ...


## 2. Index the chunks (Bi-Encoder + FAISS)
Vectors are L2-normalised, so the inner product equals cosine similarity.

In [3]:
bi_encoder = SentenceTransformer("BAAI/bge-small-en-v1.5")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

t0 = time.perf_counter()
emb = bi_encoder.encode([c["text"] for c in chunks], normalize_embeddings=True, batch_size=64)
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
print(f"indexed {index.ntotal} chunks, dim={emb.shape[1]}, took {time.perf_counter()-t0:.2f}s")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10415.88it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 11279.63it/s]

indexed 700 chunks, dim=384, took 0.98s


## 3. The two pipelines
* `baseline_retrieve` returns the **Top-10** by cosine similarity (Pipeline 1 uses the first 3).
* `rerank` scores the same 10 (query, chunk) pairs with the Cross-Encoder and sorts them descending (Pipeline 2 uses the first 3).
* `generate` is shared, so Answer 1 and Answer 2 come from an *identical* prompt structure.

BGE recommends a query instruction prefix for retrieval queries, so it is used.

In [4]:
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def baseline_retrieve(query, k=10):
    """Pipeline 1, stage 1: dense retrieval. Returns [(chunk_id, cosine_score), ...] best first."""
    q = bi_encoder.encode([QUERY_PREFIX + query], normalize_embeddings=True)
    scores, ids = index.search(q, k)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]

def rerank(query, candidates):
    """Pipeline 2: score each (query, chunk) pair jointly, then re-sort. Returns the full re-sorted list."""
    ids = [cid for cid, _ in candidates]
    ce_scores = cross_encoder.predict([(query, chunks[i]["text"]) for i in ids])
    order = np.argsort(-ce_scores)
    return [(ids[o], float(ce_scores[o])) for o in order]

PROMPT = """You are answering questions about NVIDIA's fiscal 2024 annual report.
Use ONLY the context below. If the context does not contain the answer, say "The context does not say."
Answer in at most 3 sentences.

Context:
{context}

Question: {question}
Answer:"""

def generate(query, chunk_ids):
    ctx = "\n\n".join(f"[Chunk {i}, p.{chunks[i]['page']}] {chunks[i]['text']}" for i in chunk_ids)
    r = ollama.chat(model=LLM, messages=[{"role": "user", "content": PROMPT.format(context=ctx, question=query)}],
                    options={"temperature": 0, "seed": 0})
    return r["message"]["content"].strip()

## 4. Failure-mode stress test
For each hand-written query I read the PDF and noted a **gold phrase** that identifies the evidence chunk, plus the **key words a correct answer needs**. The notebook then records:

* the gold chunk's rank in the baseline Top-10 and after re-ranking (if it is not in the Top-10 at all, its rank over the *whole* index is shown instead),
* whether it reached the Top-3 that the LLM sees,
* whether Answer 1 / Answer 2 are correct (a rough automatic check; a refusal such as "The context does not say" never counts as correct. I still read every answer).

Besides the cases that work, the list includes *controls* (nothing should change) and **negative results** (the evidence is never retrieved, so re-ranking the Top-10 cannot help). Cases where re-ranking *demotes* evidence are counted in the synthetic benchmark (section 4.3). Together this keeps the comparison from being cherry-picked.

In [5]:
# (category, query, regex identifying the gold chunk, regex a correct answer must match)
TESTS = [
 ("Exception / corner case",
  "Are there any exceptions for NEOs regarding their participation in the ESPP?",
  r"unless otherwise prohibited by the rules of the Internal Revenue", r"Internal Revenue|IRS"),
 ("Negation",
  "What did not change in Fiscal 2024 compared to Fiscal 2023 for NEOs?",
  r"No Changes to Total Target Pay", r"target pay|total target"),
 ("Keyword trap (name appears in many chunks)",
  "On what date did John O. Dabiri adopt the trading arrangement for selling NVIDIA common stock?",
  r"Dabiri, a member of our Board of Directors, adopted", r"December 18, 2023"),
 ("Buried chunk (rank 4-10)",
  "During which Fall season did the company contact its top institutional holders for stockholder outreach?",
  r"During the Fall of 2022", r"2022"),
 ("Buried chunk (rank 4-10)",
  "Under what conditions are documents available free of charge on your website?",
  r"available free of charge on or through our website", r"as soon as reasonably practicable|after .{0,40}(file|furnish)"),
 ("Control (already rank 1)",
  "What share of total revenue came from the single largest customer?",
  r"Customer A, represented 13%", r"13\s?%"),
 ("Control (fact stated in several places)",
  "In fiscal year 2024, how much was distributed to shareholders through quarterly cash dividends?",
  r"paid \$395 million in quarterly cash dividends", r"395"),
 ("Negative result: negation, evidence never retrieved",
  "What is NOT allowed under the insider trading policy?",
  r"prohibits hedging, pledging", r"hedg|pledg|margin|derivativ"),
 ("Negative result: exception, evidence never retrieved",
  "Which directors on the board are not independent?",
  r"independent, except for our CEO", r"CEO|Huang"),
]

In [6]:
def is_correct(ans, ans_re):
    # rough automatic check: a refusal ("The context does not say...") never counts as correct,
    # otherwise the answer must contain the expected key words
    if re.match(r"\s*the context does not say", ans, re.I):
        return False
    return bool(re.search(ans_re, ans, re.I))

def full_dense_rank(query, chunk_id):
    """Rank of a chunk when the bi-encoder ranks the ENTIRE corpus (used when it misses the Top-10)."""
    q = bi_encoder.encode([QUERY_PREFIX + query], normalize_embeddings=True)
    _, ids = index.search(q, len(chunks))
    return int(np.where(ids[0] == chunk_id)[0][0]) + 1

rows, answers = [], {}
for cat, q, gold_re, ans_re in TESTS:
    base10 = baseline_retrieve(q, 10)
    rr10   = rerank(q, base10)
    base_ids, rr_ids = [c for c, _ in base10], [c for c, _ in rr10]
    gold_all = [c["id"] for c in chunks if re.search(gold_re, c["text"])]       # every chunk containing the evidence
    gold_top = [i for i in base_ids if i in gold_all]                           # ...that made the Top-10
    best = min(gold_top, key=base_ids.index) if gold_top else None
    ans1, ans2 = generate(q, base_ids[:3]), generate(q, rr_ids[:3])
    answers[q] = (ans1, ans2)
    rows.append({
        "category": cat, "query": q,
        "gold_chunk": best if best is not None else gold_all[0],
        "base_rank": base_ids.index(best) + 1 if best is not None else None,
        "rerank_rank": rr_ids.index(best) + 1 if best is not None else None,
        "dense_rank_if_missed": None if best is not None else full_dense_rank(q, gold_all[0]),
        "gold_in_top3_base": best is not None and base_ids.index(best) < 3,
        "gold_in_top3_rerank": best is not None and rr_ids.index(best) < 3,
        "answer1_ok": is_correct(ans1, ans_re), "answer2_ok": is_correct(ans2, ans_re),
    })
results = pd.DataFrame(rows)
results["shift"] = results.apply(lambda r: f"#{int(r.base_rank)} -> #{int(r.rerank_rank)}" if pd.notna(r.base_rank)
                                 else f"not in Top-10 (dense rank #{int(r.dense_rank_if_missed)})", axis=1)
pd.set_option("display.max_colwidth", 60, "display.width", 250)
results[["category", "query", "gold_chunk", "shift", "gold_in_top3_base", "gold_in_top3_rerank", "answer1_ok", "answer2_ok"]]

,category,query,gold_chunk,shift,gold_in_top3_base,gold_in_top3_rerank,answer1_ok,answer2_ok
0,Exception / corner case,Are there any exceptions for NEOs regarding their partic...,259,#7 -> #1,False,True,False,True
1,Negation,What did not change in Fiscal 2024 compared to Fiscal 20...,211,#4 -> #1,False,True,True,True
2,Keyword trap (name appears in many chunks),On what date did John O. Dabiri adopt the trading arrang...,571,#6 -> #1,False,True,False,True
3,Buried chunk (rank 4-10),During which Fall season did the company contact its top...,224,#6 -> #1,False,True,False,False
4,Buried chunk (rank 4-10),Under what conditions are documents available free of ch...,394,#7 -> #1,False,True,False,True
5,Control (already rank 1),What share of total revenue came from the single largest...,549,#1 -> #1,True,True,True,True
6,Control (fact stated in several places),"In fiscal year 2024, how much was distributed to shareho...",560,#1 -> #1,True,True,True,True
7,"Negative result: negation, evidence never retrieved",What is NOT allowed under the insider trading policy?,268,not in Top-10 (dense rank #110),False,False,False,False
8,"Negative result: exception, evidence never retrieved",Which directors on the board are not independent?,63,not in Top-10 (dense rank #51),False,False,False,False


### 4.1 Answer 1 (baseline) vs. Answer 2 (re-ranked), in full

In [7]:
for (cat, q, *_), (_, row) in zip(TESTS, results.iterrows()):
    a1, a2 = answers[q]
    print("=" * 110)
    print(f"[{cat}]  {q}")
    print(f"  evidence chunk {row['gold_chunk']} (p.{chunks[row['gold_chunk']]['page']}): {row['shift']}")
    print(f"\n  ANSWER 1 (baseline top-3):\n    {a1}")
    print(f"\n  ANSWER 2 (re-ranked top-3):\n    {a2}\n")

[Exception / corner case]  Are there any exceptions for NEOs regarding their participation in the ESPP?
  evidence chunk 259 (p.82): #7 -> #1

  ANSWER 1 (baseline top-3):
    The context does not say.

  ANSWER 2 (re-ranked top-3):
    No exceptions were mentioned for NEOs regarding their participation in the ESPP. The context states that all NEOs are eligible to participate in the ESPP unless otherwise prohibited by the rules of the Internal Revenue Service, which is the same basis as other employees.

[Negation]  What did not change in Fiscal 2024 compared to Fiscal 2023 for NEOs?
  evidence chunk 211 (p.72): #4 -> #1

  ANSWER 1 (baseline top-3):
    Base salaries and variable cash opportunities for NEOs remained unchanged in Fiscal 2024 compared to Fiscal 2023. The target pay levels were kept flat, with no intended increases in target equity opportunities.

  ANSWER 2 (re-ranked top-3):
    Base salaries and variable cash opportunities did not change for NEOs between Fiscal 2023 a

### 4.2 Sanity check: what the evidence chunks actually say

In [8]:
for (cat, q, gold_re, _), r in zip(TESTS, rows):
    c = chunks[r["gold_chunk"]]
    m = re.search(gold_re, c["text"]); s = max(0, m.start() - 120)
    print(f"chunk {c['id']} (p.{c['page']}) <- {q}\n   ...{c['text'][s:m.end()+160]}...\n")

chunk 259 (p.82) <- Are there any exceptions for NEOs regarding their participation in the ESPP?
   ... NEOs on the same basis as our other employees. Like other employees, our NEOs are eligible to participate in our ESPP, unless otherwise prohibited by the rules of the Internal Revenue Service, and our 401(k) plan, which included a Company match of salary deferral contributions of up to $9,000 for calendar 2023 and up to $11,500 for calendar ...

chunk 211 (p.72) <- What did not change in Fiscal 2024 compared to Fiscal 2023 for NEOs?
   ... on annual revenue performance, in addition to base salary. Additional SY PSU Opportunities and Lower Threshold Payout; No Changes to Total Target Pay Amounts The CC designed Fiscal 2024 NEO total target pay to be flat with Fiscal 2023, but adjusted certain features of the compensation program to motivate our...

chunk 571 (p.145) <- On what date did John O. Dabiri adopt the trading arrangement for selling NVIDIA common stock?
   ... instances of fr

## 4.3 Larger synthetic benchmark (to avoid judging from a handful of hand-picked queries)
For 260 randomly chosen prose chunks the local LLM writes one question that the chunk answers. The question style rotates through four types: **paraphrase** (avoid the passage's own words), **negation**, **exception/condition**, and **plain factual**. The source chunk is the gold label. Questions are cached in `data/synthetic_questions.json` so reruns are identical.

*Caveat:* the questions are LLM-written, so a few are odd (some mention "the passage"), and the same fact is sometimes stated in two places in a 10-K (MD&A and the notes), so a hit on the duplicate counts as a miss. This slightly under-counts both pipelines equally.

In [9]:
SYNTH_PATH = "data/synthetic_questions.json"
STYLES = ["paraphrase", "negation", "exception", "factual"]
INSTR = ["Write a question that paraphrases the passage's ideas WITHOUT reusing its distinctive words (use synonyms).",
         "Write a question involving a negation or prohibition (what is NOT allowed / what does NOT happen / what is excluded).",
         "Write a question about an exception or special condition (except, unless, only if) stated in the passage.",
         "Write a specific factual question whose answer is a detail stated in the passage."]

if os.path.exists(SYNTH_PATH):
    synth = json.load(open(SYNTH_PATH))
else:                                          # (re)generate: ~3 minutes on a local GPU
    prose = [c for c in chunks if sum(ch.isdigit() for ch in c["text"]) / len(c["text"]) < 0.04
             and len(c["text"].split()) >= 150]          # skip number-heavy table chunks
    random.seed(0)
    synth = []
    for k, c in enumerate(random.sample(prose, 260)):
        p = (f"Passage:\n{c['text']}\n\n{INSTR[k % 4]} The question must be answerable from this passage alone "
             "and must not mention 'the passage'. Output only the question.")
        r = ollama.chat(model=LLM, messages=[{"role": "user", "content": p}], options={"temperature": 0.3, "seed": k})
        synth.append({"chunk": c["id"], "style": k % 4, "q": r["message"]["content"].strip().split("\n")[0]})
    json.dump(synth, open(SYNTH_PATH, "w"), indent=1)

for x in synth:
    base = baseline_retrieve(x["q"], 10); rr = rerank(x["q"], base)
    b, r_ = [c for c, _ in base], [c for c, _ in rr]
    x["base_rank"] = b.index(x["chunk"]) + 1 if x["chunk"] in b else None
    x["rerank_rank"] = r_.index(x["chunk"]) + 1 if x["chunk"] in b else None
syn = pd.DataFrame(synth)

def summarise(d):
    hit3 = lambda col: (d[col].notna() & (d[col] <= 3)).mean()
    mrr  = lambda col: (1 / d[col]).fillna(0).mean()
    return pd.Series({"n": len(d), "in_top10": d.base_rank.notna().mean(),
                      "top3_baseline": hit3("base_rank"), "top3_reranked": hit3("rerank_rank"),
                      "MRR_baseline": mrr("base_rank"), "MRR_reranked": mrr("rerank_rank")})

overall = summarise(syn).to_frame("all questions").T
by_style = syn.assign(style=syn["style"].map(dict(enumerate(STYLES)))).groupby("style").apply(summarise)
pd.concat([overall, by_style]).round(3)

,n,in_top10,top3_baseline,top3_reranked,MRR_baseline,MRR_reranked
all questions,260.0,0.881,0.715,0.835,0.600,0.736
exception,65.0,0.908,0.738,0.877,0.605,0.795
factual,65.0,0.908,0.662,0.846,0.543,0.736
negation,65.0,0.738,0.646,0.692,0.528,0.627
paraphrase,65.0,0.969,0.815,0.923,0.722,0.787


In [10]:
promoted = syn[(syn.base_rank >= 4) & (syn.rerank_rank <= 3)]
demoted  = syn[(syn.base_rank <= 3) & (syn.rerank_rank >= 4)]
print(f"Promoted into the Top-3 (was rank 4-10): {len(promoted)} questions")
print(f"Pushed out of the Top-3 (was rank 1-3) : {len(demoted)} questions")
print(f"Not retrieved in Top-10 at all         : {syn.base_rank.isna().sum()} questions (re-ranking cannot help these)")

Promoted into the Top-3 (was rank 4-10): 34 questions
Pushed out of the Top-3 (was rank 1-3) : 3 questions
Not retrieved in Top-10 at all         : 31 questions (re-ranking cannot help these)


## 5. Latency profiling
Each stage is timed over many repetitions **after a warm-up** (the first GPU call includes CUDA start-up and would distort the numbers). Retrieval and re-ranking are timed separately from LLM generation, which is identical in both pipelines.

In [11]:
import statistics as st
queries = [t[1] for t in TESTS]

def timeit(fn, reps=20):
    for q in queries[:2]: fn(q)                       # warm-up
    ts = []
    for _ in range(reps):
        for q in queries:
            t = time.perf_counter(); fn(q); ts.append((time.perf_counter() - t) * 1000)
    return ts

t_base = timeit(lambda q: baseline_retrieve(q, 10))
t_both = timeit(lambda q: rerank(q, baseline_retrieve(q, 10)))
t_gen = []
for q in queries:
    ids = [c for c, _ in baseline_retrieve(q, 3)]
    t = time.perf_counter(); generate(q, ids); t_gen.append((time.perf_counter() - t) * 1000)

lat = pd.DataFrame({
    "stage": ["Pipeline 1: bi-encoder retrieval (top-10)", "Pipeline 2: retrieval + cross-encoder re-rank",
              "LLM generation (same for both)"],
    "mean_ms":   [st.mean(t_base), st.mean(t_both), st.mean(t_gen)],
    "median_ms": [st.median(t_base), st.median(t_both), st.median(t_gen)],
    "p95_ms":    [np.percentile(t_base, 95), np.percentile(t_both, 95), np.percentile(t_gen, 95)],
}).round(1)
lat.loc[len(lat)] = ["=> re-ranking overhead", lat.mean_ms[1] - lat.mean_ms[0], lat.median_ms[1] - lat.median_ms[0],
                     lat.p95_ms[1] - lat.p95_ms[0]]
lat

,stage,mean_ms,median_ms,p95_ms
0,Pipeline 1: bi-encoder retrieval (top-10),5.8,5.8,5.9
1,Pipeline 2: retrieval + cross-encoder re-rank,14.8,14.7,16.8
2,LLM generation (same for both),492.7,475.8,615.9
3,=> re-ranking overhead,9.0,8.9,10.9


In [12]:
print(f"Re-ranking multiplies retrieval time by {lat.mean_ms[1]/lat.mean_ms[0]:.1f}x, "
      f"but is only {lat.mean_ms[3]/(lat.mean_ms[1]+lat.mean_ms[2])*100:.1f}% of the retrieval+generation time.")
print("device used by cross-encoder:", cross_encoder.model.device)

Re-ranking multiplies retrieval time by 2.6x, but is only 1.8% of the retrieval+generation time.
device used by cross-encoder: cuda:0


## 6. Why not run the Cross-Encoder over the whole collection?
A bi-encoder embeds every chunk **once, offline**; a query then costs one encoder pass plus a (fast, even approximate) nearest-neighbour lookup. A cross-encoder needs query and chunk **together**, so nothing can be precomputed: each query would need one full forward pass **per chunk**. The cell below times that on our own corpus and extrapolates.

In [13]:
q = queries[0]
all_pairs = [(q, c["text"]) for c in chunks]
cross_encoder.predict(all_pairs[:16])                      # warm-up
t = time.perf_counter(); cross_encoder.predict(all_pairs); t_full = (time.perf_counter() - t) * 1000
per_pair = t_full / len(chunks)
t_10 = float(np.mean(t_both) - np.mean(t_base))

print(f"Cross-encoder over ALL {len(chunks)} chunks: {t_full:.0f} ms  ({per_pair:.2f} ms/pair, batched on GPU)")
print(f"Cross-encoder over 10 candidates          : {t_10:.0f} ms")
for n in [10_000, 1_000_000, 100_000_000]:
    print(f"  extrapolated to {n:>11,} chunks: {per_pair*n/1000:>12,.1f} s per query (linear in corpus size)")

Cross-encoder over ALL 700 chunks: 344 ms  (0.49 ms/pair, batched on GPU)
Cross-encoder over 10 candidates          : 9 ms
  extrapolated to      10,000 chunks:          4.9 s per query (linear in corpus size)
  extrapolated to   1,000,000 chunks:        491.2 s per query (linear in corpus size)
  extrapolated to 100,000,000 chunks:     49,123.9 s per query (linear in corpus size)


## 7. Save results

In [14]:
results.to_csv("results_rank_shifts.csv", index=False)
lat.to_csv("results_latency.csv", index=False)
pd.concat([overall, by_style]).round(3).to_csv("results_synthetic_benchmark.csv")
json.dump({q: {"answer1_baseline": a1, "answer2_reranked": a2} for q, (a1, a2) in answers.items()},
          open("results_answers.json", "w"), indent=2)
print("saved: results_rank_shifts.csv, results_latency.csv, results_synthetic_benchmark.csv, results_answers.json")

saved: results_rank_shifts.csv, results_latency.csv, results_synthetic_benchmark.csv, results_answers.json
